In [1]:

import sqlite3
import pandas as pd

# Crear la base de datos
conexion = sqlite3.connect("features_clientes.db")

# Crear las tablas
conexion.executescript("""

DROP TABLE IF EXISTS detalles_pedido;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (
    cliente_id INTEGER PRIMARY KEY,
    nombre TEXT,
    ciudad TEXT
);

CREATE TABLE productos (
    producto_id INTEGER PRIMARY KEY,
    nombre TEXT,
    categoria TEXT
);

CREATE TABLE pedidos (
    pedido_id INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    fecha TEXT,
    FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id)
);

CREATE TABLE detalles_pedido (
    detalle_id INTEGER PRIMARY KEY,
    pedido_id INTEGER,
    producto_id INTEGER,
    cantidad INTEGER,
    precio_unitario REAL,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(pedido_id),
    FOREIGN KEY (producto_id) REFERENCES productos(producto_id)
);

""")

# Insertar clientes
conexion.executemany("""
INSERT INTO clientes VALUES (?, ?, ?)
""", [
    (101, 'Ana', 'Buenos Aires'),
    (102, 'Bruno', 'Buenos Aires'),
    (103, 'Carla', 'Cordoba'),
    (104, 'Diego', 'Cordoba'),
    (105, 'Elena', 'Buenos Aires')
])

# Insertar productos
conexion.executemany("""
INSERT INTO productos VALUES (?, ?, ?)
""", [
    (1, 'Arroz', 'Alimentos'),
    (2, 'Leche', 'Alimentos'),
    (3, 'Notebook', 'Tecnologia'),
    (4, 'Mouse', 'Tecnologia')
])

# Insertar pedidos
conexion.executemany("""
INSERT INTO pedidos VALUES (?, ?, ?)
""", [
    (1, 101, '2026-01-01'),
    (2, 101, '2026-01-05'),
    (3, 101, '2026-01-10'),
    (4, 101, '2026-01-15'),
    (5, 102, '2026-01-03'),
    (6, 102, '2026-01-08'),
    (7, 103, '2026-01-02'),
    (8, 103, '2026-01-06'),
    (9, 103, '2026-01-11'),
    (10, 103, '2026-01-16'),
    (11, 103, '2026-01-20'),
    (12, 105, '2026-01-04'),
    (13, 105, '2026-01-09'),
    (14, 105, '2026-01-14'),
    (15, 105, '2026-01-19')
])

# Insertar detalles de pedidos
conexion.executemany("""
INSERT INTO detalles_pedido
VALUES (?, ?, ?, ?, ?)
""", [
    (1, 1, 1, 2, 30),
    (2, 2, 3, 1, 120),
    (3, 3, 2, 2, 20),
    (4, 4, 4, 2, 50),
    (5, 5, 3, 1, 150),
    (6, 6, 1, 2, 30),
    (7, 7, 3, 1, 200),
    (8, 8, 1, 2, 40),
    (9, 9, 4, 2, 50),
    (10, 10, 2, 3, 20),
    (11, 11, 3, 1, 100),
    (12, 12, 1, 1, 20),
    (13, 13, 2, 1, 20),
    (14, 14, 4, 1, 20),
    (15, 15, 1, 2, 20)
])

conexion.commit()

# Comprobar las tablas creadas
for tabla in ['clientes', 'pedidos',
              'detalles_pedido', 'productos']:
    print(f"\nTabla: {tabla}")
    display(pd.read_sql_query(
        f"SELECT * FROM {tabla}", conexion
    ))



Tabla: clientes


,cliente_id,nombre,ciudad
0,101,Ana,Buenos Aires
1,102,Bruno,Buenos Aires
2,103,Carla,Cordoba
3,104,Diego,Cordoba
4,105,Elena,Buenos Aires



Tabla: pedidos


,pedido_id,cliente_id,fecha
0,1,101,2026-01-01
1,2,101,2026-01-05
2,3,101,2026-01-10
3,4,101,2026-01-15
4,5,102,2026-01-03
5,6,102,2026-01-08
6,7,103,2026-01-02
7,8,103,2026-01-06
8,9,103,2026-01-11
9,10,103,2026-01-16



Tabla: detalles_pedido


,detalle_id,pedido_id,producto_id,cantidad,precio_unitario
0,1,1,1,2,30.0
1,2,2,3,1,120.0
2,3,3,2,2,20.0
3,4,4,4,2,50.0
4,5,5,3,1,150.0
5,6,6,1,2,30.0
6,7,7,3,1,200.0
7,8,8,1,2,40.0
8,9,9,4,2,50.0
9,10,10,2,3,20.0



Tabla: productos


,producto_id,nombre,categoria
0,1,Arroz,Alimentos
1,2,Leche,Alimentos
2,3,Notebook,Tecnologia
3,4,Mouse,Tecnologia


In [2]:

# 1. Calcular el gasto total histórico por cliente (LTV)

consulta_1 = """
SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,

    -- Suma el importe total gastado por cada cliente
    COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0)
        AS gasto_total

FROM clientes c

-- Conserva todos los clientes aunque no tengan pedidos
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id

LEFT JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id

GROUP BY
    c.cliente_id,
    c.nombre,
    c.ciudad

ORDER BY gasto_total DESC;
"""

resultado_1 = pd.read_sql_query(consulta_1, conexion)

display(resultado_1)


,cliente_id,nombre,ciudad,gasto_total
0,103,Carla,Cordoba,540.0
1,101,Ana,Buenos Aires,320.0
2,102,Bruno,Buenos Aires,210.0
3,105,Elena,Buenos Aires,100.0
4,104,Diego,Cordoba,0.0


In [3]:

# 1. Calcular el gasto total histórico por cliente (LTV)

consulta_1 = """
SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,

    -- Suma el importe total gastado por cada cliente
    COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0)
        AS gasto_total

FROM clientes c

-- Conserva todos los clientes aunque no tengan pedidos
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id

LEFT JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id

GROUP BY
    c.cliente_id,
    c.nombre,
    c.ciudad

ORDER BY gasto_total DESC;
"""

resultado_1 = pd.read_sql_query(consulta_1, conexion)

display(resultado_1)


,cliente_id,nombre,ciudad,gasto_total
0,103,Carla,Cordoba,540.0
1,101,Ana,Buenos Aires,320.0
2,102,Bruno,Buenos Aires,210.0
3,105,Elena,Buenos Aires,100.0
4,104,Diego,Cordoba,0.0


In [4]:

# 3. Identificar clientes activos

consulta_3 = """
SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,

    -- Cuenta la cantidad de pedidos realizados
    COUNT(DISTINCT p.pedido_id) AS cantidad_pedidos,

    -- Calcula el gasto promedio por pedido
    ROUND(
        SUM(dp.cantidad * dp.precio_unitario) * 1.0 /
        COUNT(DISTINCT p.pedido_id), 2
    ) AS gasto_promedio,

    -- Calcula el gasto total del cliente
    SUM(dp.cantidad * dp.precio_unitario) AS gasto_total

FROM clientes c

INNER JOIN pedidos p
    ON c.cliente_id = p.cliente_id

INNER JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id

GROUP BY
    c.cliente_id,
    c.nombre,
    c.ciudad

-- Filtra los clientes según las condiciones solicitadas
HAVING
    COUNT(DISTINCT p.pedido_id) > 3
    AND
    SUM(dp.cantidad * dp.precio_unitario) * 1.0 /
    COUNT(DISTINCT p.pedido_id) > 50

ORDER BY gasto_total DESC;
"""

resultado_3 = pd.read_sql_query(consulta_3, conexion)

display(resultado_3)


,cliente_id,nombre,ciudad,cantidad_pedidos,gasto_promedio,gasto_total
0,103,Carla,Cordoba,5,108.0,540.0
1,101,Ana,Buenos Aires,4,80.0,320.0


In [5]:

# 4. Ranking de clientes por ciudad

consulta_4 = """
WITH resumen_clientes AS (

    SELECT
        c.cliente_id,
        c.nombre,
        c.ciudad,

        -- Calcula el gasto total de cada cliente
        COALESCE(
            SUM(dp.cantidad * dp.precio_unitario), 0
        ) AS gasto_total

    FROM clientes c

    LEFT JOIN pedidos p
        ON c.cliente_id = p.cliente_id

    LEFT JOIN detalles_pedido dp
        ON p.pedido_id = dp.pedido_id

    GROUP BY
        c.cliente_id,
        c.nombre,
        c.ciudad
)

SELECT
    cliente_id,
    nombre,
    ciudad,
    gasto_total,

    -- Clasifica a los clientes según su gasto
    -- reiniciando el ranking en cada ciudad
    RANK() OVER (
        PARTITION BY ciudad
        ORDER BY gasto_total DESC
    ) AS ranking_ciudad

FROM resumen_clientes

ORDER BY ciudad, ranking_ciudad;
"""

resultado_4 = pd.read_sql_query(consulta_4, conexion)

display(resultado_4)


,cliente_id,nombre,ciudad,gasto_total,ranking_ciudad
0,101,Ana,Buenos Aires,320.0,1
1,102,Bruno,Buenos Aires,210.0,2
2,105,Elena,Buenos Aires,100.0,3
3,103,Carla,Cordoba,540.0,1
4,104,Diego,Cordoba,0.0,2


In [6]:

%%writefile ejercicio_features_clientes.sql

-- =====================================================
-- EJERCICIO: FEATURES DE CLIENTES PARA MACHINE LEARNING
-- Base de datos: features_clientes
-- Motor: SQLite
-- =====================================================

PRAGMA foreign_keys = ON;

-- 1. CREACION DE TABLAS

DROP TABLE IF EXISTS detalles_pedido;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (
    cliente_id INTEGER PRIMARY KEY,
    nombre TEXT,
    ciudad TEXT
);

CREATE TABLE productos (
    producto_id INTEGER PRIMARY KEY,
    nombre TEXT,
    categoria TEXT
);

CREATE TABLE pedidos (
    pedido_id INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    fecha TEXT,
    FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id)
);

CREATE TABLE detalles_pedido (
    detalle_id INTEGER PRIMARY KEY,
    pedido_id INTEGER,
    producto_id INTEGER,
    cantidad INTEGER,
    precio_unitario REAL,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(pedido_id),
    FOREIGN KEY (producto_id) REFERENCES productos(producto_id)
);

-- 2. INSERCION DE DATOS

INSERT INTO clientes VALUES
(101, 'Ana', 'Buenos Aires'),
(102, 'Bruno', 'Buenos Aires'),
(103, 'Carla', 'Cordoba'),
(104, 'Diego', 'Cordoba'),
(105, 'Elena', 'Buenos Aires');

INSERT INTO productos VALUES
(1, 'Arroz', 'Alimentos'),
(2, 'Leche', 'Alimentos'),
(3, 'Notebook', 'Tecnologia'),
(4, 'Mouse', 'Tecnologia');

INSERT INTO pedidos VALUES
(1, 101, '2026-01-01'),
(2, 101, '2026-01-05'),
(3, 101, '2026-01-10'),
(4, 101, '2026-01-15'),
(5, 102, '2026-01-03'),
(6, 102, '2026-01-08'),
(7, 103, '2026-01-02'),
(8, 103, '2026-01-06'),
(9, 103, '2026-01-11'),
(10, 103, '2026-01-16'),
(11, 103, '2026-01-20'),
(12, 105, '2026-01-04'),
(13, 105, '2026-01-09'),
(14, 105, '2026-01-14'),
(15, 105, '2026-01-19');

INSERT INTO detalles_pedido VALUES
(1, 1, 1, 2, 30),
(2, 2, 3, 1, 120),
(3, 3, 2, 2, 20),
(4, 4, 4, 2, 50),
(5, 5, 3, 1, 150),
(6, 6, 1, 2, 30),
(7, 7, 3, 1, 200),
(8, 8, 1, 2, 40),
(9, 9, 4, 2, 50),
(10, 10, 2, 3, 20),
(11, 11, 3, 1, 100),
(12, 12, 1, 1, 20),
(13, 13, 2, 1, 20),
(14, 14, 4, 1, 20),
(15, 15, 1, 2, 20);

-- =====================================================
-- 3. CONSULTA 1: LTV (GASTO TOTAL POR CLIENTE)
-- LEFT JOIN conserva clientes sin compras.
-- COALESCE reemplaza NULL por 0.
-- =====================================================

SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,
    COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0) AS gasto_total
FROM clientes c
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id
LEFT JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
ORDER BY gasto_total DESC;

-- =====================================================
-- 4. CONSULTA 2: GASTO POR CATEGORIA
-- Agregacion condicional con CASE WHEN.
-- =====================================================

SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,
    COALESCE(SUM(
        CASE WHEN pr.categoria = 'Alimentos'
        THEN dp.cantidad * dp.precio_unitario ELSE 0 END
    ), 0) AS gasto_alimentos,
    COALESCE(SUM(
        CASE WHEN pr.categoria = 'Tecnologia'
        THEN dp.cantidad * dp.precio_unitario ELSE 0 END
    ), 0) AS gasto_tecnologia,
    COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0) AS gasto_total
FROM clientes c
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id
LEFT JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id
LEFT JOIN productos pr
    ON dp.producto_id = pr.producto_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
ORDER BY c.cliente_id;

-- =====================================================
-- 5. CONSULTA 3: SEGMENTACION DE CLIENTES ACTIVOS
-- Mas de 3 pedidos y gasto promedio superior a 50.
-- HAVING filtra despues de la agrupacion.
-- =====================================================

SELECT
    c.cliente_id,
    c.nombre,
    c.ciudad,
    COUNT(DISTINCT p.pedido_id) AS cantidad_pedidos,
    ROUND(
        SUM(dp.cantidad * dp.precio_unitario) * 1.0 /
        COUNT(DISTINCT p.pedido_id), 2
    ) AS gasto_promedio,
    SUM(dp.cantidad * dp.precio_unitario) AS gasto_total
FROM clientes c
INNER JOIN pedidos p
    ON c.cliente_id = p.cliente_id
INNER JOIN detalles_pedido dp
    ON p.pedido_id = dp.pedido_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
HAVING COUNT(DISTINCT p.pedido_id) > 3
   AND SUM(dp.cantidad * dp.precio_unitario) * 1.0 /
       COUNT(DISTINCT p.pedido_id) > 50
ORDER BY gasto_total DESC;

-- =====================================================
-- 6. CONSULTA 4: RANKING DE CLIENTES POR CIUDAD
-- CTE para calcular el gasto antes de aplicar RANK.
-- =====================================================

WITH resumen_clientes AS (
    SELECT
        c.cliente_id,
        c.nombre,
        c.ciudad,
        COALESCE(SUM(dp.cantidad * dp.precio_unitario), 0) AS gasto_total
    FROM clientes c
    LEFT JOIN pedidos p
        ON c.cliente_id = p.cliente_id
    LEFT JOIN detalles_pedido dp
        ON p.pedido_id = dp.pedido_id
    GROUP BY c.cliente_id, c.nombre, c.ciudad
)

SELECT
    cliente_id,
    nombre,
    ciudad,
    gasto_total,
    RANK() OVER (
        PARTITION BY ciudad
        ORDER BY gasto_total DESC
    ) AS ranking_ciudad
FROM resumen_clientes
ORDER BY ciudad, ranking_ciudad;

-- =====================================================
-- FIN DEL EJERCICIO
-- =====================================================


Writing ejercicio_features_clientes.sql


In [7]:
from google.colab import files
files.download('ejercicio_features_clientes.sql')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>